# Montana Child Tax Credit Reform Analysis (2027)

This notebook analyzes the impact of a Montana Child Tax Credit reform.

## Reform Details
- **Credit Amount**: $1,800 per child under age 6, $1,500 per child ages 6-17
- **Eligibility**: Children under 18 who are dependents with SSN
- **Earned Income Requirement**: At least $1 of earned income required (enabled by default)
- **Phase-out thresholds**:
  - Joint/Surviving Spouse: $120,000 AGI
  - Single/Head of Household/Separate: $60,000 AGI
- **Phase-out**: $50 reduction per $1,000 of income above threshold
- **Refundable**: Yes

## Analysis Goals
1. Budgetary impact
2. Winners and losers
3. Income decile analysis
4. Poverty impacts (overall, child, deep)
5. Income bracket breakdown

In [1]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np

MT_DATASET = "hf://policyengine/policyengine-us-data/states/MT.h5"
PERIOD = 2027

# Intra-decile bounds and labels (matches API v2)
INTRA_BOUNDS = [-np.inf, -0.05, -1e-3, 1e-3, 0.05, np.inf]
INTRA_LABELS = [
    "Lose more than 5%",
    "Lose less than 5%",
    "No change",
    "Gain less than 5%",
    "Gain more than 5%",
]

## Define Reform

In [2]:
def create_mt_ctc_reform():
    """Enable Montana Child Tax Credit reform for 2027."""
    reform = Reform.from_dict(
        {
            "gov.contrib.states.mt.ctc.in_effect": {
                "2027-01-01.2100-12-31": True
            }
        },
        country_id="us",
    )
    return reform

print("Reform function defined!")

Reform function defined!


## Load Simulations

In [3]:
print("Loading baseline (current law - no MT CTC)...")
baseline = Microsimulation(dataset=MT_DATASET)
print("Baseline loaded")

print("\nLoading reform (MT CTC enabled)...")
reform = create_mt_ctc_reform()
reformed = Microsimulation(dataset=MT_DATASET, reform=reform)
print("Reform loaded")

print("\n" + "="*60)
print("All simulations ready!")
print("="*60)

Loading baseline (current law - no MT CTC)...
Baseline loaded

Loading reform (MT CTC enabled)...
Reform loaded

All simulations ready!


## Helper Functions

In [4]:
def poverty_metrics(baseline_rate, reform_rate):
    """Return rate change and percent change for a poverty metric."""
    rate_change = reform_rate - baseline_rate
    percent_change = (
        rate_change / baseline_rate * 100
        if baseline_rate > 0
        else 0.0
    )
    return rate_change, percent_change

## Budgetary Impact

In [5]:
# Calculate household-level income change
baseline_net_income = baseline.calc("household_net_income", period=PERIOD, map_to="household")
reformed_net_income = reformed.calc("household_net_income", period=PERIOD, map_to="household")
income_change = reformed_net_income - baseline_net_income

# Total cost
total_cost = float(income_change.sum())

# Total households
total_households = float((income_change * 0 + 1).sum())

# Direct credit calculation for validation
mt_ctc_total = float(reformed.calc("mt_ctc", period=PERIOD).sum())
tax_units_receiving = float((reformed.calc("mt_ctc", period=PERIOD) > 0).sum())

print("="*60)
print(f"BUDGETARY IMPACT ({PERIOD})")
print("="*60)
print(f"Total cost:                    ${total_cost/1e6:,.2f} million")
print(f"Total households:              {total_households:,.0f}")
print(f"\nDirect credit amount:          ${mt_ctc_total/1e6:,.2f} million")
print(f"Tax units receiving credit:    {tax_units_receiving:,.0f}")
print(f"Average credit per tax unit:   ${mt_ctc_total / tax_units_receiving:,.0f}")
print(f"\nValidation ratio:              {total_cost / mt_ctc_total:.2f}x")
print("="*60)

BUDGETARY IMPACT (2027)
Total cost:                    $257.54 million
Total households:              410,732

Direct credit amount:          $257.58 million
Tax units receiving credit:    94,504
Average credit per tax unit:   $2,726

Validation ratio:              1.00x


## Winners and Losers

In [6]:
# Person-level winners/losers (consistent with intra-decile)
# Use same weighting approach as intra-decile: people_per_hh * household_weight
household_weight = reformed.calc("household_weight", period=PERIOD)
people_per_hh = baseline.calc("household_count_people", period=PERIOD, map_to="household")

weight_arr = np.array(household_weight)
change_arr = np.array(income_change)
people_arr = np.array(people_per_hh)

# Person-weighted totals
people_weighted = people_arr * weight_arr
total_people = float(people_weighted.sum())

# Winners/losers using person-weighted counts
winners_mask = change_arr > 1
losers_mask = change_arr < -1
beneficiaries_mask = change_arr > 0

winners = float(people_weighted[winners_mask].sum())
losers = float(people_weighted[losers_mask].sum())
beneficiaries = float(people_weighted[beneficiaries_mask].sum())

winners_rate = winners / total_people * 100
losers_rate = losers / total_people * 100

# Average benefit for affected households
affected_mask = np.abs(change_arr) > 1
affected_count = float(weight_arr[affected_mask].sum())
avg_benefit = (
    float(np.average(
        change_arr[affected_mask],
        weights=weight_arr[affected_mask],
    ))
    if affected_count > 0
    else 0.0
)

print("="*60)
print(f"WINNERS AND LOSERS ({PERIOD})")
print("="*60)
print(f"Winners (gain > $1):           {winners:,.0f} ({winners_rate:.1f}%)")
print(f"Losers (lose > $1):            {losers:,.0f} ({losers_rate:.1f}%)")
print(f"Beneficiaries (gain > $0):     {beneficiaries:,.0f}")
print(f"\nAverage benefit (affected):    ${avg_benefit:,.0f}")
print("="*60)

WINNERS AND LOSERS (2027)
Winners (gain > $1):           415,596 (35.9%)
Losers (lose > $1):            0 (0.0%)
Beneficiaries (gain > $0):     415,596

Average benefit (affected):    $2,809


## Households and Children

In [7]:
# Calculate households and children metrics
age = baseline.calc("age", period=PERIOD)
is_child_person = np.array(age) < 18

# Household-level: count children per household
children_per_hh = baseline.calc("tax_unit_count_dependents", period=PERIOD, map_to="household")
children_per_hh_arr = np.array(children_per_hh)
has_children = children_per_hh_arr > 0

# Total households with children
hh_with_children = float(weight_arr[has_children].sum())

# Households with children that benefit
hh_with_children_benefit = float(weight_arr[has_children & (change_arr > 1)].sum())
hh_with_children_benefit_pct = hh_with_children_benefit / hh_with_children * 100

# Total children (using person weights)
person_weight = baseline.calc("person_weight", period=PERIOD)
pw_arr = np.array(person_weight)
total_children = float((pw_arr * is_child_person).sum())

# Children benefiting (map income change to person level)
income_change_person = reformed.calc("household_net_income", period=PERIOD, map_to="person") - \
                       baseline.calc("household_net_income", period=PERIOD, map_to="person")
income_change_person_arr = np.array(income_change_person)
children_benefit_mask = is_child_person & (income_change_person_arr > 1)
children_benefiting = float((pw_arr * children_benefit_mask).sum())
children_benefit_pct = children_benefiting / total_children * 100

print("="*60)
print(f"HOUSEHOLDS AND CHILDREN ({PERIOD})")
print("="*60)
print(f"\nHOUSEHOLDS WITH CHILDREN:")
print(f"  Total households with children:      {hh_with_children:,.0f}")
print(f"  Households with children benefiting: {hh_with_children_benefit:,.0f}")
print(f"  Percentage benefiting:               {hh_with_children_benefit_pct:.1f}%")
print(f"\nCHILDREN (under 18):")
print(f"  Total children:                      {total_children:,.0f}")
print(f"  Children benefiting:                 {children_benefiting:,.0f}")
print(f"  Percentage benefiting:               {children_benefit_pct:.1f}%")
print("="*60)

HOUSEHOLDS AND CHILDREN (2027)

HOUSEHOLDS WITH CHILDREN:
  Total households with children:      144,771
  Households with children benefiting: 91,687
  Percentage benefiting:               63.3%

CHILDREN (under 18):
  Total children:                      241,202
  Children benefiting:                 187,908
  Percentage benefiting:               77.9%


## Income Decile Analysis

In [8]:
# Get decile assignment
decile = baseline.calc("household_income_decile", period=PERIOD, map_to="household")

decile_average = {}
decile_relative = {}

for d in range(1, 11):
    dmask = decile == d
    d_count = float(dmask.sum())
    if d_count > 0:
        d_change_sum = float(income_change[dmask].sum())
        decile_average[str(d)] = d_change_sum / d_count
        d_baseline_sum = float(baseline_net_income[dmask].sum())
        decile_relative[str(d)] = (
            d_change_sum / d_baseline_sum * 100
            if d_baseline_sum != 0
            else 0.0
        )
    else:
        decile_average[str(d)] = 0.0
        decile_relative[str(d)] = 0.0

print("="*60)
print(f"INCOME DECILE ANALYSIS ({PERIOD})")
print("="*60)
print(f"{'Decile':<10} {'Avg Change':>15} {'Relative Change':>18}")
print("-"*45)
for d in range(1, 11):
    print(f"{d:<10} ${decile_average[str(d)]:>13,.0f} {decile_relative[str(d)]:>17.2f}%")
print("="*60)

INCOME DECILE ANALYSIS (2027)
Decile          Avg Change    Relative Change
---------------------------------------------
1          $          121              0.63%
2          $          214              0.54%
3          $          486              0.87%
4          $          927              1.35%
5          $          710              0.87%
6          $        1,217              1.26%
7          $        1,331              1.16%
8          $          921              0.67%
9          $          799              0.44%
10         $          264              0.03%


## Intra-Decile Distribution

In [9]:
# Intra-decile requires person-weighted proportions
people_per_hh = baseline.calc("household_count_people", period=PERIOD, map_to="household")
capped_baseline = np.maximum(np.array(baseline_net_income), 1)
rel_change_arr = np.array(income_change) / capped_baseline

decile_arr = np.array(decile)
people_weighted = np.array(people_per_hh) * weight_arr

intra_decile_deciles = {label: [] for label in INTRA_LABELS}
for d in range(1, 11):
    dmask = decile_arr == d
    d_people = people_weighted[dmask]
    d_total_people = d_people.sum()
    d_rel = rel_change_arr[dmask]

    for lower, upper, label in zip(
        INTRA_BOUNDS[:-1], INTRA_BOUNDS[1:], INTRA_LABELS
    ):
        in_group = (d_rel > lower) & (d_rel <= upper)
        proportion = (
            float(d_people[in_group].sum() / d_total_people)
            if d_total_people > 0
            else 0.0
        )
        intra_decile_deciles[label].append(proportion)

intra_decile_all = {
    label: sum(intra_decile_deciles[label]) / 10
    for label in INTRA_LABELS
}

print("="*60)
print(f"INTRA-DECILE DISTRIBUTION ({PERIOD})")
print("="*60)
print("Overall distribution:")
for label in INTRA_LABELS:
    print(f"  {label:<20} {intra_decile_all[label]*100:>6.1f}%")
print("="*60)

INTRA-DECILE DISTRIBUTION (2027)
Overall distribution:
  Lose more than 5%       0.0%
  Lose less than 5%       0.0%
  No change              64.1%
  Gain less than 5%      27.6%
  Gain more than 5%       8.3%


## Poverty Impact

In [10]:
# Overall poverty
pov_bl = baseline.calc("in_poverty", period=PERIOD, map_to="person")
pov_rf = reformed.calc("in_poverty", period=PERIOD, map_to="person")
poverty_baseline_rate = float(pov_bl.mean() * 100)
poverty_reform_rate = float(pov_rf.mean() * 100)
poverty_rate_change, poverty_percent_change = poverty_metrics(
    poverty_baseline_rate, poverty_reform_rate
)

# Deep poverty
deep_bl = baseline.calc("in_deep_poverty", period=PERIOD, map_to="person")
deep_rf = reformed.calc("in_deep_poverty", period=PERIOD, map_to="person")
deep_poverty_baseline_rate = float(deep_bl.mean() * 100)
deep_poverty_reform_rate = float(deep_rf.mean() * 100)
deep_poverty_rate_change, deep_poverty_percent_change = poverty_metrics(
    deep_poverty_baseline_rate, deep_poverty_reform_rate
)

# Child poverty requires age filtering
age_arr = np.array(baseline.calc("age", period=PERIOD))
is_child = age_arr < 18
pw_arr = np.array(baseline.calc("person_weight", period=PERIOD))
child_w = pw_arr[is_child]
total_child_w = child_w.sum()

pov_bl_arr = np.array(pov_bl).astype(bool)
pov_rf_arr = np.array(pov_rf).astype(bool)
deep_bl_arr = np.array(deep_bl).astype(bool)
deep_rf_arr = np.array(deep_rf).astype(bool)

def child_rate(arr):
    return float(
        (arr[is_child] * child_w).sum() / total_child_w * 100
    ) if total_child_w > 0 else 0.0

child_poverty_baseline_rate = child_rate(pov_bl_arr)
child_poverty_reform_rate = child_rate(pov_rf_arr)
child_poverty_rate_change, child_poverty_percent_change = poverty_metrics(
    child_poverty_baseline_rate, child_poverty_reform_rate
)

deep_child_poverty_baseline_rate = child_rate(deep_bl_arr)
deep_child_poverty_reform_rate = child_rate(deep_rf_arr)
deep_child_poverty_rate_change, deep_child_poverty_percent_change = poverty_metrics(
    deep_child_poverty_baseline_rate, deep_child_poverty_reform_rate
)

print("="*60)
print(f"POVERTY IMPACT ({PERIOD})")
print("="*60)
print("\nOVERALL POVERTY:")
print(f"  Baseline rate:     {poverty_baseline_rate:.2f}%")
print(f"  Reform rate:       {poverty_reform_rate:.2f}%")
print(f"  Rate change:       {poverty_rate_change:+.2f} pp")
print(f"  Percent change:    {poverty_percent_change:+.1f}%")

print("\nDEEP POVERTY:")
print(f"  Baseline rate:     {deep_poverty_baseline_rate:.2f}%")
print(f"  Reform rate:       {deep_poverty_reform_rate:.2f}%")
print(f"  Rate change:       {deep_poverty_rate_change:+.2f} pp")
print(f"  Percent change:    {deep_poverty_percent_change:+.1f}%")

print("\nCHILD POVERTY (under 18):")
print(f"  Baseline rate:     {child_poverty_baseline_rate:.2f}%")
print(f"  Reform rate:       {child_poverty_reform_rate:.2f}%")
print(f"  Rate change:       {child_poverty_rate_change:+.2f} pp")
print(f"  Percent change:    {child_poverty_percent_change:+.1f}%")

print("\nDEEP CHILD POVERTY (under 18):")
print(f"  Baseline rate:     {deep_child_poverty_baseline_rate:.2f}%")
print(f"  Reform rate:       {deep_child_poverty_reform_rate:.2f}%")
print(f"  Rate change:       {deep_child_poverty_rate_change:+.2f} pp")
print(f"  Percent change:    {deep_child_poverty_percent_change:+.1f}%")
print("="*60)

POVERTY IMPACT (2027)

OVERALL POVERTY:
  Baseline rate:     14.19%
  Reform rate:       13.57%
  Rate change:       -0.61 pp
  Percent change:    -4.3%

DEEP POVERTY:
  Baseline rate:     6.10%
  Reform rate:       6.01%
  Rate change:       -0.09 pp
  Percent change:    -1.4%

CHILD POVERTY (under 18):
  Baseline rate:     12.84%
  Reform rate:       11.35%
  Rate change:       -1.49 pp
  Percent change:    -11.6%

DEEP CHILD POVERTY (under 18):
  Baseline rate:     5.49%
  Reform rate:       5.26%
  Rate change:       -0.23 pp
  Percent change:    -4.3%


## Income Bracket Breakdown

In [11]:
# Get AGI at household level
agi = reformed.calc("adjusted_gross_income", period=PERIOD, map_to="household")
agi_arr = np.array(agi)
affected_mask = np.abs(change_arr) > 1

income_brackets = [
    (0, 25_000, "Under $25k"),
    (25_000, 50_000, "$25k-$50k"),
    (50_000, 75_000, "$50k-$75k"),
    (75_000, 100_000, "$75k-$100k"),
    (100_000, 150_000, "$100k-$150k"),
    (150_000, float("inf"), "Over $150k"),
]

by_income_bracket = []
for min_inc, max_inc, label in income_brackets:
    mask = (
        (agi_arr >= min_inc)
        & (agi_arr < max_inc)
        & affected_mask
    )
    bracket_affected = float(weight_arr[mask].sum())
    if bracket_affected > 0:
        bracket_cost = float(
            (change_arr[mask] * weight_arr[mask]).sum()
        )
        bracket_avg = float(
            np.average(change_arr[mask], weights=weight_arr[mask])
        )
    else:
        bracket_cost = 0.0
        bracket_avg = 0.0
    by_income_bracket.append({
        "bracket": label,
        "beneficiaries": bracket_affected,
        "total_cost": bracket_cost,
        "avg_benefit": bracket_avg,
    })

print("="*60)
print(f"INCOME BRACKET BREAKDOWN ({PERIOD})")
print("="*60)
print(f"{'Bracket':<15} {'Beneficiaries':>15} {'Total Cost':>15} {'Avg Benefit':>12}")
print("-"*60)
for row in by_income_bracket:
    print(f"{row['bracket']:<15} {row['beneficiaries']:>15,.0f} ${row['total_cost']/1e6:>13,.2f}M ${row['avg_benefit']:>10,.0f}")
print("="*60)

INCOME BRACKET BREAKDOWN (2027)
Bracket           Beneficiaries      Total Cost  Avg Benefit
------------------------------------------------------------
Under $25k                8,736 $        24.16M $     2,765
$25k-$50k                12,669 $        31.20M $     2,463
$50k-$75k                16,468 $        45.74M $     2,777
$75k-$100k               18,437 $        55.78M $     3,025
$100k-$150k              22,656 $        69.10M $     3,050
Over $150k               12,217 $        30.17M $     2,469


## Summary

In [12]:
print("\n" + "="*70)
print("MONTANA CHILD TAX CREDIT REFORM SUMMARY")
print("="*70)
print(f"\nYear: {PERIOD}")
print(f"\nReform: Refundable credit - $1,800 per child under 6, $1,500 per child 6-17")
print(f"        Phase-out starts at $60k (single) / $120k (joint) AGI")
print(f"        Phase-out rate: $50 per $1,000 above threshold")
print(f"        Earned income requirement: at least $1")
print("\n" + "-"*70)
print("KEY FINDINGS:")
print("-"*70)
print(f"\n  BUDGET:")
print(f"    Total cost:                    ${total_cost/1e6:,.2f} million")
print(f"    Total households:              {total_households:,.0f}")
print(f"    Total people:                  {total_people:,.0f}")
print(f"\n  DISTRIBUTION (person-level):")
print(f"    Winners:                       {winners:,.0f} ({winners_rate:.1f}%)")
print(f"    Losers:                        {losers:,.0f} ({losers_rate:.1f}%)")
print(f"    Average benefit (per HH):      ${avg_benefit:,.0f}")
print(f"\n  POVERTY REDUCTION:")
print(f"    Overall poverty:               {poverty_rate_change:+.2f} pp ({poverty_percent_change:+.1f}%)")
print(f"    Deep poverty:                  {deep_poverty_rate_change:+.2f} pp ({deep_poverty_percent_change:+.1f}%)")
print(f"    Child poverty:                 {child_poverty_rate_change:+.2f} pp ({child_poverty_percent_change:+.1f}%)")
print(f"    Deep child poverty:            {deep_child_poverty_rate_change:+.2f} pp ({deep_child_poverty_percent_change:+.1f}%)")
print("="*70)


MONTANA CHILD TAX CREDIT REFORM SUMMARY

Year: 2027

Reform: Refundable credit - $1,800 per child under 6, $1,500 per child 6-17
        Phase-out starts at $60k (single) / $120k (joint) AGI
        Phase-out rate: $50 per $1,000 above threshold
        Earned income requirement: at least $1

----------------------------------------------------------------------
KEY FINDINGS:
----------------------------------------------------------------------

  BUDGET:
    Total cost:                    $257.54 million
    Total households:              410,732
    Total people:                  1,156,438

  DISTRIBUTION (person-level):
    Winners:                       415,596 (35.9%)
    Losers:                        0 (0.0%)
    Average benefit (per HH):      $2,809

  POVERTY REDUCTION:
    Overall poverty:               -0.61 pp (-4.3%)
    Deep poverty:                  -0.09 pp (-1.4%)
    Child poverty:                 -1.49 pp (-11.6%)
    Deep child poverty:            -0.23 pp (-4.3%)

## Export Results

In [13]:
# Compile all results into a dictionary
results = {
    "budget": {
        "total_cost": total_cost,
        "households": total_households,
        "people": total_people,
    },
    "decile": {
        "average": decile_average,
        "relative": decile_relative,
    },
    "intra_decile": {
        "all": intra_decile_all,
        "deciles": intra_decile_deciles,
    },
    "winners": winners,
    "losers": losers,
    "winners_rate": winners_rate,
    "losers_rate": losers_rate,
    "beneficiaries": beneficiaries,
    "avg_benefit": avg_benefit,
    "poverty_baseline_rate": poverty_baseline_rate,
    "poverty_reform_rate": poverty_reform_rate,
    "poverty_rate_change": poverty_rate_change,
    "poverty_percent_change": poverty_percent_change,
    "child_poverty_baseline_rate": child_poverty_baseline_rate,
    "child_poverty_reform_rate": child_poverty_reform_rate,
    "child_poverty_rate_change": child_poverty_rate_change,
    "child_poverty_percent_change": child_poverty_percent_change,
    "deep_poverty_baseline_rate": deep_poverty_baseline_rate,
    "deep_poverty_reform_rate": deep_poverty_reform_rate,
    "deep_poverty_rate_change": deep_poverty_rate_change,
    "deep_poverty_percent_change": deep_poverty_percent_change,
    "deep_child_poverty_baseline_rate": deep_child_poverty_baseline_rate,
    "deep_child_poverty_reform_rate": deep_child_poverty_reform_rate,
    "deep_child_poverty_rate_change": deep_child_poverty_rate_change,
    "deep_child_poverty_percent_change": deep_child_poverty_percent_change,
    "by_income_bracket": by_income_bracket,
}

# Export summary to CSV
summary_df = pd.DataFrame([
    {"Metric": "Total cost", "Value": f"${total_cost/1e6:,.2f}M"},
    {"Metric": "Total households", "Value": f"{total_households:,.0f}"},
    {"Metric": "Total people", "Value": f"{total_people:,.0f}"},
    {"Metric": "Winners (people)", "Value": f"{winners:,.0f} ({winners_rate:.1f}%)"},
    {"Metric": "Losers (people)", "Value": f"{losers:,.0f} ({losers_rate:.1f}%)"},
    {"Metric": "Average benefit (per HH)", "Value": f"${avg_benefit:,.0f}"},
    {"Metric": "Overall poverty change", "Value": f"{poverty_rate_change:+.2f} pp ({poverty_percent_change:+.1f}%)"},
    {"Metric": "Child poverty change", "Value": f"{child_poverty_rate_change:+.2f} pp ({child_poverty_percent_change:+.1f}%)"},
    {"Metric": "Deep poverty change", "Value": f"{deep_poverty_rate_change:+.2f} pp ({deep_poverty_percent_change:+.1f}%)"},
    {"Metric": "Deep child poverty change", "Value": f"{deep_child_poverty_rate_change:+.2f} pp ({deep_child_poverty_percent_change:+.1f}%)"},
])
print(summary_df.to_string(index=False))

summary_df.to_csv("mt_ctc_results.csv", index=False)
print("\nExported to: mt_ctc_results.csv")

                   Metric             Value
               Total cost          $257.54M
         Total households           410,732
             Total people         1,156,438
         Winners (people)   415,596 (35.9%)
          Losers (people)          0 (0.0%)
 Average benefit (per HH)            $2,809
   Overall poverty change  -0.61 pp (-4.3%)
     Child poverty change -1.49 pp (-11.6%)
      Deep poverty change  -0.09 pp (-1.4%)
Deep child poverty change  -0.23 pp (-4.3%)

Exported to: mt_ctc_results.csv
